In [1]:
"""
Method 5: Lottery Ticket Hypothesis — ResNet-50 / CIFAR-10
===========================================================
The Lottery Ticket Hypothesis (Frankle & Carlin, 2019):
  "A randomly initialized dense network contains a sparse subnetwork
   (the winning ticket) that can match full network performance."

Algorithm (Iterative Magnitude Pruning):
  1. Prune iteratively to target sparsity → ticket mask
  2. Apply mask to trained weights         → winning ticket
  3. Apply same mask to random weights     → random-init control
  
Both variants are evaluated and reported. The LTH advantage
(winning_acc − random_acc) validates the hypothesis.

Saves exactly 8 metrics per sparsity level (winning ticket) for comparison
with baseline: accuracy, precision, recall, f1, num_parameters,
model_size_mb, inference_time_ms, flops

Output: method5_lottery_ticket_pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "method5_lottery_ticket_pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS  = [0.30, 0.50, 0.70, 0.80, 0.90]
ITERATIVE_ROUNDS = 5    # rounds to reach target sparsity iteratively
INFERENCE_RUNS   = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── LTH Helpers ───────────────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def find_ticket_mask(trained_model, target_sparsity, n_rounds):
    """
    Iterative magnitude pruning to find the winning ticket mask.
    Per-round rate computed so n_rounds reaches target_sparsity:
      (1 − per_round)^n_rounds = (1 − target_sparsity)
    """
    per_round = 1.0 - (1.0 - target_sparsity) ** (1.0 / n_rounds)
    current   = copy.deepcopy(trained_model)

    for _ in range(n_rounds):
        layers = prunable_layers(current)
        prune.global_unstructured(layers, pruning_method=prune.L1Unstructured,
                                  amount=per_round)
        for module, param in layers:
            prune.remove(module, param)

    return current

def extract_mask(model):
    """Binary mask: 1 = surviving weight, 0 = pruned."""
    return {name: (module.weight.data != 0).float()
            for name, module in model.named_modules()
            if isinstance(module, (nn.Conv2d, nn.Linear))}

def apply_mask(source_model, mask_dict, random_init=False):
    """
    Apply ticket mask to source_model.
    random_init=False → keep trained weights, zero pruned positions.
    random_init=True  → kaiming-normal re-init, zero pruned positions.
    """
    result = copy.deepcopy(source_model)
    for name, module in result.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and name in mask_dict:
            m = mask_dict[name].to(DEVICE)
            if random_init:
                nn.init.kaiming_normal_(module.weight.data,
                                        mode="fan_out", nonlinearity="relu")
            module.weight.data *= m
    return result


# ── 8 Core Metrics ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / (1024 ** 2)
    finally:
        os.remove(tmp)
    return size

def measure_inference_time_ms(model):
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32).to(DEVICE)

    if DEVICE.type == "cuda":
        for _ in range(20):
            model(dummy)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / INFERENCE_RUNS
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        return (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return int(macs * 2)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*62}")
    print("  Method 5: Lottery Ticket Hypothesis — ResNet-50 / CIFAR-10")
    print(f"  Device : {DEVICE}  |  IMP rounds: {ITERATIVE_ROUNDS}")
    print(f"{'='*62}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    results = []

    for sparsity in SPARSITY_LEVELS:
        print(f"\n  Target sparsity: {int(sparsity*100)}%")

        # Find ticket mask via iterative magnitude pruning
        print(f"    Finding ticket mask ({ITERATIVE_ROUNDS} IMP rounds)...")
        ticket_model = find_ticket_mask(model, sparsity, ITERATIVE_ROUNDS)
        mask         = extract_mask(ticket_model)

        # Winning ticket: trained weights × mask
        print("    Evaluating: winning ticket (trained weights × mask)...")
        winning = apply_mask(model, mask, random_init=False)
        m_win   = evaluate(winning, loader)

        # Random-init control: kaiming-normal weights × same mask
        print("    Evaluating: random-init control (random weights × mask)...")
        random  = apply_mask(model, mask, random_init=True)
        m_rnd   = evaluate(random, loader)

        num_params    = count_parameters(winning)
        model_size_mb = get_model_size_mb(winning)
        inference_ms  = measure_inference_time_ms(winning)
        flops         = compute_flops(winning)

        print(f"    Winning acc: {m_win['accuracy']:.4f} | "
              f"Random acc: {m_rnd['accuracy']:.4f} | "
              f"LTH advantage: {(m_win['accuracy']-m_rnd['accuracy'])*100:.2f}%")

        row = {
            "sparsity"        : sparsity,
            "iterative_rounds": ITERATIVE_ROUNDS,
            # ── 8 standardized metrics (winning ticket) ──
            "accuracy"         : round(m_win["accuracy"],  6),
            "precision"        : round(m_win["precision"], 6),
            "recall"           : round(m_win["recall"],    6),
            "f1"               : round(m_win["f1"],        6),
            "num_parameters"   : num_params,
            "model_size_mb"    : round(model_size_mb, 4),
            "inference_time_ms": round(inference_ms, 4),
            "flops"            : flops,
            # ── LTH-specific extras ──
            "random_init_accuracy": round(m_rnd["accuracy"], 6),
            "lth_advantage_pct"   : round((m_win["accuracy"] - m_rnd["accuracy"]) * 100, 4),
        }
        results.append(row)

    report = {
        "method"     : "lottery_ticket_hypothesis",
        "description": ("LTH: iterative magnitude pruning to find ticket mask. "
                        "Primary metrics are for the winning ticket (trained weights × mask). "
                        "random_init_accuracy and lth_advantage_pct are LTH-specific extras."),
        "baseline"   : baseline,
        "results"    : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


  Method 5: Lottery Ticket Hypothesis — ResNet-50 / CIFAR-10
  Device : cuda  |  IMP rounds: 5


  Target sparsity: 30%
    Finding ticket mask (5 IMP rounds)...
    Evaluating: winning ticket (trained weights × mask)...
    Evaluating: random-init control (random weights × mask)...
    Winning acc: 0.9321 | Random acc: 0.1000 | LTH advantage: 83.21%

  Target sparsity: 50%
    Finding ticket mask (5 IMP rounds)...
    Evaluating: winning ticket (trained weights × mask)...
    Evaluating: random-init control (random weights × mask)...
    Winning acc: 0.9321 | Random acc: 0.0963 | LTH advantage: 83.58%

  Target sparsity: 70%
    Finding ticket mask (5 IMP rounds)...
    Evaluating: winning ticket (trained weights × mask)...
    Evaluating: random-init control (random weights × mask)...
    Winning acc: 0.9321 | Random acc: 0.1000 | LTH advantage: 83.21%

  Target sparsity: 80%
    Finding ticket mask (5 IMP rounds)...
    Evaluating: winning ticket (trained weights × mask)...
    Eva